# PerturbDiff on GSE132080

This notebook follows the preprocessing pattern used in `Squidiff_reproducibility`: build a clean AnnData object first, perform QC and gene selection, save a prepared `.h5ad`, then train PerturbDiff from that prepared file.

See `preprocessing_workflow.md` in this folder for the rationale and expected data contract.

In [ ]:
from pathlib import Path
import sys

import gzip

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.io import mmread
from sklearn.decomposition import TruncatedSVD
import torch
from torch.utils.data import DataLoader, Subset

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "reproducibility":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from perturbdiff import DiffusionConfig, ModelConfig, PerturbDiffusionModel
from perturbdiff.data import load_h5ad_perturbation_dataset, read_metadata_table

print(PROJECT_ROOT)

In [ ]:
DATA_DIR = PROJECT_ROOT / "datasets" / "GSE132080"
METADATA_PATH = DATA_DIR / "GSE132080_cell_identities.csv.gz"
PREPARED_DIR = PROJECT_ROOT / "reproducibility" / "prepared"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
PREPARED_H5AD = PREPARED_DIR / "gse132080_preprocessed.h5ad"

PERTURBATION_COL_RAW = "guide_identity"
PERTURBATION_COL_CLEAN = "perturbation_label"
BARCODE_COL = "cell_barcode"

MIN_COUNTS = 1000
MIN_GENES = 200
MAX_GENES = 10000
MAX_PCT_MT = 20.0
MIN_CELLS_PER_GENE = 3
N_TOP_GENES = 1000

MAX_TRAIN_CELLS = 2048
BATCH_SIZE = 64
EPOCHS = 1
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR, METADATA_PATH, PREPARED_H5AD, DEVICE

## Read 10x Data And Metadata

The matrix is loaded with SciPy and stored as an AnnData object. Cells are rows and genes are columns. Metadata are then aligned by barcode.

In [ ]:
def read_tsv_gz(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as handle:
        return [line.rstrip("\n").split("\t") for line in handle]

matrix = mmread(DATA_DIR / "matrix.mtx.gz").tocsr().transpose().tocsr()
barcodes = [row[0] for row in read_tsv_gz(DATA_DIR / "barcodes.tsv.gz")]
gene_rows = read_tsv_gz(DATA_DIR / "genes.tsv.gz")
gene_names = [row[1] if len(row) > 1 else row[0] for row in gene_rows]

obs = pd.DataFrame(index=pd.Index(barcodes, name="barcode"))
var = pd.DataFrame(index=pd.Index(gene_names, name="gene"))
adata = ad.AnnData(X=matrix, obs=obs, var=var)
adata.var_names_make_unique()

metadata_rows = read_metadata_table(METADATA_PATH)
metadata_by_barcode = {row[BARCODE_COL]: row for row in metadata_rows if row.get(BARCODE_COL)}
matched_barcodes = [bc for bc in adata.obs_names if bc in metadata_by_barcode]
adata = adata[matched_barcodes].copy()

for key in metadata_rows[0].keys():
    adata.obs[key] = [metadata_by_barcode[bc].get(key, "") for bc in adata.obs_names]

print(adata)
adata.obs[[BARCODE_COL, PERTURBATION_COL_RAW]].head()

## Clean Perturbation Labels And Basic Cell Filters

Controls are collapsed to `control`. Non-control guide labels are converted to target-level labels by taking the token before the first underscore.

In [ ]:
def clean_perturbation_label(value):
    text = str(value)
    lowered = text.lower()
    if "neg_ctrl" in lowered or "non-targeting" in lowered or "non_targeting" in lowered:
        return "control"
    return text.split("_", 1)[0]

adata.obs[PERTURBATION_COL_CLEAN] = [clean_perturbation_label(v) for v in adata.obs[PERTURBATION_COL_RAW]]

if "good_coverage" in adata.obs:
    adata = adata[adata.obs["good_coverage"].astype(str).str.lower().isin(["true", "1", "yes"])].copy()
if "number_of_cells" in adata.obs:
    adata = adata[adata.obs["number_of_cells"].astype(float) <= 1].copy()

print(adata)
adata.obs[PERTURBATION_COL_CLEAN].value_counts().head(15)

## QC, Normalization, And Gene Selection

The thresholds below are editable. Raw counts are preserved in `adata.layers['counts']`; the model trains on normalized/log-transformed highly variable genes.

In [ ]:
def as_1d(array_like):
    return np.asarray(array_like).reshape(-1)

X = adata.X.tocsr() if sparse.issparse(adata.X) else sparse.csr_matrix(adata.X)
counts_per_cell = as_1d(X.sum(axis=1))
genes_per_cell = as_1d((X > 0).sum(axis=1))
mt_mask = adata.var_names.str.upper().str.startswith("MT-")
mt_counts = as_1d(X[:, mt_mask].sum(axis=1)) if mt_mask.any() else np.zeros(X.shape[0])

adata.obs["total_counts"] = counts_per_cell
adata.obs["n_genes_by_counts"] = genes_per_cell
adata.obs["pct_counts_mt"] = np.divide(mt_counts, counts_per_cell, out=np.zeros_like(mt_counts), where=counts_per_cell > 0) * 100

cell_mask = (
    (adata.obs["total_counts"] >= MIN_COUNTS)
    & (adata.obs["n_genes_by_counts"] >= MIN_GENES)
    & (adata.obs["n_genes_by_counts"] <= MAX_GENES)
    & (adata.obs["pct_counts_mt"] < MAX_PCT_MT)
)
adata = adata[cell_mask.to_numpy()].copy()
X = adata.X.tocsr()

gene_cells = as_1d((X > 0).sum(axis=0))
non_mt_ribo = ~adata.var_names.str.upper().str.startswith(("MT-", "RPS", "RPL"))
gene_mask = (gene_cells >= MIN_CELLS_PER_GENE) & non_mt_ribo
adata = adata[:, gene_mask].copy()
X = adata.X.tocsr()

adata.layers["counts"] = X.copy()
cell_sums = as_1d(X.sum(axis=1))
scale = np.divide(1e4, cell_sums, out=np.zeros_like(cell_sums), where=cell_sums > 0)
X_norm = sparse.diags(scale).dot(X).tocsr()
X_norm.data = np.log1p(X_norm.data)
adata.X = X_norm

means = as_1d(X_norm.mean(axis=0))
sq_means = as_1d(X_norm.power(2).mean(axis=0))
variances = np.maximum(sq_means - means**2, 0.0)
dispersion = variances / np.maximum(means, 1e-12)
hvg_idx = np.argsort(dispersion)[-min(N_TOP_GENES, X_norm.shape[1]):]
hvg_mask = np.zeros(X_norm.shape[1], dtype=bool)
hvg_mask[hvg_idx] = True

target_genes = set(adata.obs.loc[adata.obs[PERTURBATION_COL_CLEAN] != "control", PERTURBATION_COL_CLEAN])
keep_targets = adata.var_names.isin(target_genes)
adata.var["highly_variable"] = hvg_mask | keep_targets
adata = adata[:, adata.var["highly_variable"].to_numpy()].copy()

print(adata)
print("target genes kept:", int(keep_targets.sum()))

## Optional PCA QC View

This mirrors Squidiff's reproducibility notebooks at a lightweight level. It is for inspection only and is not required by the model.

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=7)
adata.obsm["X_pca2"] = svd.fit_transform(adata.X)

summary = adata.obs[PERTURBATION_COL_CLEAN].value_counts().head(20)
print(summary)
print("explained variance:", svd.explained_variance_ratio_.sum())

## Save Prepared h5ad

The training step below reads this file, not the raw 10x matrix.

In [ ]:
adata.write(PREPARED_H5AD)
PREPARED_H5AD

## Load Prepared Data For PerturbDiff

In [ ]:
dataset, perturbation_mapping = load_h5ad_perturbation_dataset(
    PREPARED_H5AD,
    perturbation_col=PERTURBATION_COL_CLEAN,
    target_strategy="raw",
)

if MAX_TRAIN_CELLS is not None:
    train_dataset = Subset(dataset, range(min(MAX_TRAIN_CELLS, len(dataset))))
else:
    train_dataset = dataset

loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
batch = next(iter(loader))

print(f"cells: {len(dataset)}")
print(f"genes: {dataset.n_genes}")
print(f"perturbation targets: {len(perturbation_mapping)}")
print(batch["x0"].shape, batch["perturbation_ids"].shape)
print(list(perturbation_mapping.items())[:10])

## Train A Small Smoke Model

In [ ]:
cfg = ModelConfig(
    n_genes=dataset.n_genes,
    n_perturbations=len(perturbation_mapping),
    latent_dim=32,
    encoder_hidden_dims=(128,),
    denoiser_hidden_dims=(256,),
    strength_hidden_dims=(64,),
    diffusion=DiffusionConfig(timesteps=100),
)

model = PerturbDiffusionModel(cfg).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
sum(p.numel() for p in model.parameters())

In [ ]:
model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    n_batches = 0
    for batch in loader:
        x0 = batch["x0"].to(DEVICE)
        perturbation_ids = batch["perturbation_ids"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        out = model.training_losses(x0, perturbation_ids)
        out["loss"].backward()
        optimizer.step()

        total_loss += float(out["loss"].detach().cpu())
        n_batches += 1

    print(f"epoch={epoch} loss={total_loss / max(n_batches, 1):.6f}")

## Save Checkpoint And Sample

In [ ]:
checkpoint_dir = PROJECT_ROOT / "runs" / "gse132080_notebook"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_config": cfg,
        "perturbation_mapping": perturbation_mapping,
    },
    checkpoint_dir / "checkpoint.pt",
)

batch = next(iter(loader))
target_names = list(perturbation_mapping.keys())[:4]
target_ids = torch.tensor([perturbation_mapping[name] for name in target_names], device=DEVICE)
x_ctrl = batch["x0"][: len(target_names)].to(DEVICE)
strengths = torch.full((len(target_names),), 0.5, device=DEVICE)

model.eval()
with torch.no_grad():
    samples = model.sample_counterfactual(x_ctrl=x_ctrl, target_ids=target_ids, strengths=strengths, steps=10)

print(checkpoint_dir / "checkpoint.pt")
print(target_names)
print(samples.shape, samples.mean().item(), samples.std().item())

## Full Evaluation: x0 + Response Proxy Run

This section records the full 1000-epoch run that uses `prediction_type="x0"` and the PS-inspired `response_proxy` column. It is intentionally separate from the small smoke model above: the smoke model checks that the notebook can train end to end, while this section loads the real checkpoint and reproduces evaluation summaries.


In [ ]:
import json
import subprocess
from dataclasses import asdict, is_dataclass

FULL_RUN_DIR = PROJECT_ROOT / "runs" / "gse132080_x0_response_proxy_1000ep"
FULL_CHECKPOINT = FULL_RUN_DIR / "checkpoint_best.pt"
FULL_METRICS_JSON = FULL_RUN_DIR / "metrics.json"
RESPONSE_PROXY_SPLIT_DIR = PROJECT_ROOT / "reproducibility" / "prepared" / "splits_response_proxy"
RESPONSE_PROXY_SUMMARY_JSON = RESPONSE_PROXY_SPLIT_DIR / "response_proxy_summary.json"

state = torch.load(FULL_CHECKPOINT, map_location="cpu", weights_only=False)
train_metrics = json.loads(FULL_METRICS_JSON.read_text())
response_proxy_summary = json.loads(RESPONSE_PROXY_SUMMARY_JSON.read_text())
model_config = state["model_config"]
model_config_record = asdict(model_config) if is_dataclass(model_config) else str(model_config)

training_record = {
    "run_dir": str(FULL_RUN_DIR),
    "checkpoint": str(FULL_CHECKPOINT),
    "train_path": str(RESPONSE_PROXY_SPLIT_DIR / "gse132080_train.h5ad"),
    "val_path": str(RESPONSE_PROXY_SPLIT_DIR / "gse132080_val.h5ad"),
    "test_path": str(RESPONSE_PROXY_SPLIT_DIR / "gse132080_test.h5ad"),
    "epochs_requested": len(train_metrics.get("history", [])),
    "best_epoch": train_metrics["best_epoch"],
    "best_val_loss": train_metrics["best_val_loss"],
    "test_metrics": train_metrics["test_metrics"],
    "used_response_proxy_col": train_metrics["used_response_proxy_col"],
    "device": train_metrics["device"],
    "model_config": model_config_record,
    "response_proxy_parameters": response_proxy_summary["parameters"],
}

training_record


### Generation Sanity Check

The first check asks whether DDIM samples live on the same expression scale as the real normalized/log-transformed data. This catches the earlier epsilon-prediction failure mode where samples exploded to hundreds while real expression values were around `0-7`.


In [ ]:
RECOMPUTE_EVALUATION = False
SANITY_JSON = PROJECT_ROOT / "runs" / "generation_sanity" / "gse132080_x0_response_proxy_1000ep_generation_sanity.json"

if RECOMPUTE_EVALUATION or not SANITY_JSON.exists():
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "generation_sanity_check.py"),
            "--checkpoint", str(FULL_CHECKPOINT),
            "--output-json", str(SANITY_JSON),
            "--device", DEVICE,
            "--batch-size", "64",
            "--ddim-steps", "50",
        ],
        check=True,
    )

sanity = json.loads(SANITY_JSON.read_text())
generation_scale = pd.DataFrame(
    [
        {"source": "real", **sanity["real_expression_stats"]},
        {"source": "ddim_control", **sanity["ddim_from_pure_noise"]["control_condition"]},
        {"source": "ddim_target", **sanity["ddim_from_pure_noise"]["target_condition"]},
        {"source": "target_strength", **sanity["strength_stats_for_target_on_control_batch"]},
    ]
)
generation_scale


### scGen And Baseline Comparison

The comparison includes four predictors on the same random train/test split:

- `no_change`: train control mean for every target.
- `train_target_mean`: train-set mean of the same target, a strong upper/reference baseline for seen perturbations.
- `scGen`: VAE latent vector arithmetic baseline.
- `PerturbDiff`: the current x0 + response_proxy checkpoint.

Metrics are computed on all genes and on train-derived top signature genes for each target. The signature-gene view is usually more informative than all-gene Pearson because most genes barely change.


In [ ]:
COMPARE_JSON = PROJECT_ROOT / "runs" / "gse132080_x0_response_proxy_scgen_compare" / "comparison_metrics.json"

if RECOMPUTE_EVALUATION or not COMPARE_JSON.exists():
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "compare_with_scgen.py"),
            "--train-path", str(RESPONSE_PROXY_SPLIT_DIR / "gse132080_train.h5ad"),
            "--test-path", str(RESPONSE_PROXY_SPLIT_DIR / "gse132080_test.h5ad"),
            "--perturbdiff-checkpoint", str(FULL_CHECKPOINT),
            "--output-json", str(COMPARE_JSON),
            "--device", DEVICE,
            "--ddim-steps", "50",
            "--scgen-batch-size", "256",
            "--signature-top-genes", "50",
        ],
        check=True,
    )

comparison = json.loads(COMPARE_JSON.read_text())
summary = comparison["summary"]
all_gene_summary = pd.DataFrame(
    [
        {
            "method": "no_change",
            "mean_pearson_r2": summary["no_change_mean_pearson_r2"],
            "mean_mse": summary["no_change_mean_mse"],
        },
        {
            "method": "train_target_mean",
            "mean_pearson_r2": summary["train_target_mean_pearson_r2"],
            "mean_mse": summary["train_target_mean_mse"],
        },
        {
            "method": "scGen",
            "mean_pearson_r2": summary["scgen_mean_pearson_r2"],
            "mean_mse": summary["scgen_mean_mse"],
        },
        {
            "method": "PerturbDiff",
            "mean_pearson_r2": summary["perturbdiff_mean_pearson_r2"],
            "mean_mse": summary["perturbdiff_mean_mse"],
        },
    ]
)
signature_summary = pd.DataFrame(summary["signature_metrics"]).T.reset_index(names="method")
all_gene_summary, signature_summary


In [ ]:
per_target_rows = []
for row in comparison["per_target"]:
    per_target_rows.append(
        {
            "target": row["target"],
            "n_test_cells": row["n_test_cells"],
            "perturbdiff_r2": row["perturbdiff"]["pearson_r2"],
            "perturbdiff_mse": row["perturbdiff"]["mse"],
            "scgen_r2": row["scgen"]["pearson_r2"],
            "train_target_mean_r2": row["train_target_mean"]["pearson_r2"],
            "no_change_r2": row["no_change"]["pearson_r2"],
            "signature_perturbdiff_r2": row["signature_metrics"].get("perturbdiff", {}).get("pearson_r2", np.nan),
            "signature_scgen_r2": row["signature_metrics"].get("scgen", {}).get("pearson_r2", np.nan),
        }
    )
per_target = pd.DataFrame(per_target_rows)
per_target.sort_values("perturbdiff_r2").head(10)
